In [ ]:
import requests
import pandas as pd
import pyodbc
from datetime import datetime, timedelta
import time

# ==============================
# 🔑 API CONFIG
# ==============================
API_KEY = "5f1e340a-3ff8-45a6-b918-64af2631037e"
URL = f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}"

# ==============================
# 🟡 SQL CONNECTION
# ==============================
conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=FARAZ-PC\\SQLEXPRESS03;"
    "DATABASE=IPL_Data;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

# ==============================
# 🔴 MAIN LOOP
# ==============================
while True:
    try:
        response = requests.get(URL)
        data = response.json()

        matches = data.get("data", [])

        if not matches:
            print("❌ No data found")
            time.sleep(20)
            continue

        records = []

        # ==============================
        # 🟢 PROCESS MATCHES
        # ==============================
        for match in matches:
            try:
                match_name = match.get("name", "")

                # ✅ IPL FILTER
                if "IPL" not in match_name and "Indian Premier League" not in match_name:
                    continue

                # ✅ STATUS LOGIC
                if not match.get("matchStarted", False):
                    status = "Upcoming"
                elif match.get("matchEnded", False):
                    status = "Finished"
                else:
                    status = "Live"

                print(f"🏏 {status} IPL MATCH:", match_name)

                match_id = match.get("id")
                score_list = match.get("score", [])

                # ✅ MATCH START TIME
                try:
                    start_time = datetime.strptime(match.get("dateTimeGMT"), "%Y-%m-%dT%H:%M:%S")
                except:
                    start_time = datetime.now()

                for i, inning in enumerate(score_list):

                    # ✅ TEAM NAME CLEAN
                    team_name = inning.get("inning", "Unknown")
                    if "Inning" in team_name:
                        team_name = team_name.split("Inning")[0].strip()

                    team_name = team_name.replace("Benguluru", "Bengaluru")

                    # ✅ SAFE VALUES
                    runs = int(inning.get("r", 0))
                    wickets = int(inning.get("w", 0))
                    overs = float(inning.get("o", 0))

                    # ✅ MATCH TIME (SIMULATED TIMELINE)
                    match_time = start_time + timedelta(minutes=i * 10)

                    record = {
                        "match_id": match_id,
                        "team": team_name,
                        "runs": runs,
                        "wickets": wickets,
                        "overs": overs,
                        "batsman": "Unknown",
                        "non_striker": "Unknown",
                        "striker_flag": 0,
                        "timestamp": datetime.now(),   # insert time
                        "match_time": match_time,      # real timeline
                        "match_status": status
                    }

                    records.append(record)

            except Exception as e:
                print("⚠️ Match parsing error:", e)

        # ==============================
        # 🟡 DATAFRAME
        # ==============================
        df = pd.DataFrame(records)

        if df.empty:
            print("❌ No IPL matches found")
            time.sleep(20)
            continue

        print("\n📊 Data Preview:")
        print(df.head())

        # ==============================
        # 🟣 INSERT INTO SQL
        # ==============================
        for _, row in df.iterrows():
            cursor.execute("""
                INSERT INTO IPL_Live_Data 
                (match_id, team, runs, wickets, overs, batsman, non_striker, striker_flag, timestamp, match_time, match_status)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            row['match_id'],
            row['team'],
            row['runs'],
            row['wickets'],
            row['overs'],
            row['batsman'],
            row['non_striker'],
            row['striker_flag'],
            row['timestamp'],
            row['match_time'],
            row['match_status']
            )

        conn.commit()
        print("✅ Data inserted successfully")

    except Exception as e:
        print("❌ Error:", e)

    # ==============================
    # ⏱️ REFRESH
    # ==============================
    time.sleep(20)

❌ No data found
❌ No data found
❌ No data found
❌ No data found
❌ No data found
